In [44]:
#Import required modules
import os
import arcpy
from arcpy import env  
from arcpy.sa import *

# Set processing environment, matching the folder location of the repository.
home_folder = r"C:\Users\Lieutenant\EGM722_Coursework_AlexMason_B01039763" # Change this file path to match your repository location.
 
# Set the extent of the processing environment using a feature class, defined by the user and saved in 'InputDataAOI' folder as a shapefile 'HLS_AOI.shp'.
process_extent = r"C:\Users\Lieutenant\EGM722_Coursework_AlexMason_B01039763\InputDataAOI\HLS_AOI.shp" # Input the file location of the HLS_AOI shapefile here.

# Set coordinate system to match the input DEMs and LULC, in this workflow case as British National Grid - BNG:
coords = r"C:\Users\Lieutenant\EGM722_Coursework_AlexMason_B01039763\BNG.prj"


Reclassify the Land Use / Land Cover (LULC) dataset, based upon the user-defined LULC reclassification table in the instruction document.

In [45]:
print('Running LULC Reclassification.')

def LULC_Reclassify():
    # Set processing environment, defined above as process_extent, matching the file path of the HLS_AOI polygon shapefile.
    arcpy.env.extent = process_extent
    
    # Set processing environment, defined above as home_folder, matching the repository file path.
    env.workspace = home_folder

    # Set local variables:
    # Set location of the input LULC dataset.
    inRasterLULC = r"\InputDataLULC\LCM.tif"
    
    # Set the field or band of the raster which holds the numerical land classes.
    reclassFieldLULC = "Value"

    # Set the reclassification table as a list of tuples, based on the reclassification table in the instruction document.
    remapLULC = RemapValue([[1,0],[2,0],[3,2],[4,3],[5,3],[6,3],[7,3],[8,0],[9,0],[10,1],[11,0],[12,1],[13,0],[14,0],[15,0],[16,2],[17,1],[18,1],[19,1],[20,0],[21,0]])

    # Execute Reclassify tool
    outReclassifyLULC = Reclassify(inRasterLULC, reclassFieldLULC, remapLULC, "NODATA")

    # Save the output 
    outReclassifyLULC.save("outReclassifyLULC.tif")

LULC_Reclassify()

print('LULC Reclassification complete')

Running LULC Reclassification.
LULC Reclassification complete


Take the set of LiDAR Composite DTM 2m tiles and mosaic them to a new Raster dataset (.tif). Ouput to match the parent coordinate system (BNG), horizontal resolution (2m), and radiometric resolution (32-bit float). Output a single raster covering only the User-Defined AOI polygon.

In [46]:
print('Running Mosaic DEMs.')

def Mosaic_DEM():  #Mosaic the DEM tiles to a New Raster covering only the AOI polygon area.
      
    # Set processing environment, defined above as process_extent, matching the file path of the HLS_AOI polygon shapefile.
    arcpy.env.extent = process_extent
        
    # Set processing environment, matching the folder location of the input DEM raster tiles 'InputDataDEM'.
    arcpy.env.workspace = r"C:\Users\Lieutenant\EGM722_Coursework_AlexMason_B01039763\InputDataDEM"

    # Get and print a list of TIFs from the workspace
    inputDEMs = arcpy.ListRasters("*", "TIF")
    for raster in inputDEMs:
            print(raster)

    # Set output raster folder location:
    # outputDEM_Folder = r"C:\Users\Lieutenant\EGM722_Coursework_AlexMason_B01039763"
    outputDEM_Folder = home_folder

    # Set Ouput DEM file name:
    outputDEM_File = "HLS_AOI_2m_DEM.tif"

    # Set coordinate system to match the input DEMs, coords is defined in script header:
    DEM_CoordinateSystem = coords
    
    # Set pixel type, matching the National LiDAR Programme input tiles, 32 bit float.
    pixel_type = "32_BIT_FLOAT"

    # Set the cell size of the output raster, to match the input DEM tiles, which for these dataset(s) is 2m pixels:
    cellsize = "2"

    # Set the number of bands for the output raster, just one, to record the elevation values:
    number_of_bands = "1"

    # Set the mosic method for the operation:
    mosaic_method = "LAST" # this is also the default option for the tool.
    
    # Set the Mosaic Colourmap Mode:
    mosaic_colormap_mode = "LAST" # This is the default for the tool.

    # Mosaic several TIFF images to a new TIFF image:
    arcpy.management.MosaicToNewRaster(inputDEMs, outputDEM_Folder, outputDEM_File, DEM_CoordinateSystem, pixel_type, cellsize, number_of_bands, mosaic_method, mosaic_colormap_mode)

Mosaic_DEM() # Run the Mosaic DEM function.

print('Mosaic DEMs complete.')

Running Mosaic DEMs.
ST84ne_DTM_2m.tif
ST84se_DTM_2m.tif
ST85ne_DTM_2m.tif
ST85se_DTM_2m.tif
ST94ne_DTM_2m.tif
ST94nw_DTM_2m.tif
ST94se_DTM_2m.tif
ST94sw_DTM_2m.tif
ST95ne_DTM_2m.tif
ST95nw_DTM_2m.tif
ST95se_DTM_2m.tif
ST95sw_DTM_2m.tif
SU04ne_DTM_2m.tif
SU04nw_DTM_2m.tif
SU04se_DTM_2m.tif
SU04sw_DTM_2m.tif
SU05ne_DTM_2m.tif
SU05nw_DTM_2m.tif
SU05se_DTM_2m.tif
SU05sw_DTM_2m.tif
SU14ne_DTM_2m.tif
SU14nw_DTM_2m.tif
SU14se_DTM_2m.tif
SU14sw_DTM_2m.tif
SU15ne_DTM_2m.tif
SU15nw_DTM_2m.tif
SU15se_DTM_2m.tif
SU15sw_DTM_2m.tif
SU24ne_DTM_2m.tif
SU24nw_DTM_2m.tif
SU24se_DTM_2m.tif
SU24sw_DTM_2m.tif
SU25ne_DTM_2m.tif
SU25nw_DTM_2m.tif
SU25se_DTM_2m.tif
SU25sw_DTM_2m.tif
Mosaic DEMs complete.


Analyse the SLOPE of the 2m DTM, using the Slope tool. Output a raster TIF with one band recording the Slope of the terrain in Degrees. Coordinate system, horizontal resolution and vertical datum matches the input DTM. Then resample the Slope to 10m pixel size and match the geometry to the reclassified LULC dataset.

In [47]:
print('Running Slope calculation.')

def Slope_2m(): # This function uses the Slope tool to first calculate Slope of the DTM in Degrees, 
                # with horizontal pixel resolution matching the input DTM.
                            
    # Set processing environment, matching the folder location of the input 2m DTM raster 'HLS_AOI_2m_DEM.tif'.
    arcpy.env.workspace = home_folder

    # Pick up the name of the 2m DTM raster from the Mosaic function:
    inRaster = "HLS_AOI_2m_DEM.tif"  

    # specify the file name of the ouput slope dataset:
    outRaster = "HLS_AOI_2m_Slope.tif" 
    
    # output the Slope as Degrees:
    outMeasurement = "DEGREE" 

    # Set the vertical scaling factor (z-factor) as 1. 
    # No scaling is required because the input DTM is horizontal units metres, and vertical units metres.
    zFactor = "1" 
    
    # Set the METHOD for the calculation based on planar (flat earth):
    method = "PLANAR"
    
    # Set the vertical Z value unit, as metres, to match the input DTM 2m raster:
    zUnit = "METER"

    # Execute Slope tool:
    arcpy.ddd.Slope(inRaster, outRaster, outMeasurement, zFactor, method, zUnit)
    
Slope_2m() # Run the Slope analysis function.

print('Slope calculation complete.')

Running Slope calculation.
Slope calculation complete.


In [48]:
print('Downsampling Slope.')

def SlopeAggregate_2to10(): # This function downsamples the Slope 2m resolution to a 10m resolution raster,
                            # while also also matching the geometry of the reclassified Land Use / Land Cover (LULC) 10m raster.
    
    # Set processing environment, matching the parent folder location for the analysis workflow, defined in script header as home_folder:
    arcpy.env.workspace = home_folder
    
    # Set the extent environment to match the LULC reclassified raster:
    arcpy.env.extent = "\outReclassifyLULC.tif"

    # Define the input raster:
    in_raster = "HLS_AOI_2m_Slope.tif"
    
    # Define the Cell Factor to agreggate by. In this case, we are aggregating from 2m to 10m so a factor of 5:
    cell_factor = "5"
    
    # Define the aggregation statistical method:
    aggregation_type = "MEAN" # This setting defines that the MEAN values of the input slope cells are calculated and recorded to the output.
    
    # Define the Extent Handling method:
    extent_handling = "EXPAND" # This setting ensure the boundaries of the output raster are expanded from the input raster, 
                                # if required, to match the geometry.
    
    # Define how to handle cells with No Data:
    ignore_nodata = "DATA" # This setting ignores pixels with NoData when aggregating to the larger pixel.
    
    # Execute Aggregate
    outSlope10m = Aggregate(in_raster, cell_factor, aggregation_type, extent_handling, ignore_nodata)

    # Save the output to the environment workspace as a raster of data type TIF: 
    outSlope10m.save("HLS_AOI_10m_Slope.tif")
    
SlopeAggregate_2to10() # Run the rasample function.

print('Slope downsampling complete.')

Downsampling Slope.
Slope downsampling complete.


The next three functions take the 10m slope raster and reclassify it based upon the slope angle requirements for landing the three different varieties of helicopter of interest (Wildcat, Merlin and Chinook). They each output integer encoded TIF rasters based upon the values defined in the tables in the Instruction Manual.

In [49]:
print('Reclassifying slope for Wildcat helicopter parameters.')

def SlopeClassifyWildcat(): # This function reclassifies the 10m slope dataset from Degrees to 
                            # numerical values representing classes defined for the Wildcat 
                            # Slope Requirements in the instruction manual document. 
    
    #Set processing environment, matching the parent folder location for the project, defined as home_folder in script header:
    env.workspace = home_folder
    
    # Set the extent of the processing environment using a feature class, defined by the user, set in the script header as process_extent: 
    arcpy.env.extent = process_extent
    
    # Set local variables:
    # Set location of the input 10m Slope dataset:
    inRasterWildcatSlope = "HLS_AOI_10m_Slope.tif"

    # Set the field or band of the raster which holds the numerical slope values in degrees:
    reclassFieldSlope = "Value"

    # Set the reclassification table as a list of tuples, based on the reclassification table in the instruction document.
    # 0 to 3 degrees = 3; 3 to 7 degrees = 2; 7 to 90 degrees = 0
    remapWildcatSlope = RemapRange([[0,3,3],[3,7,2],[7,90,0]])
    
    # Execute Reclassify tool
    outReclassifySlopeWildcat = Reclassify(inRasterWildcatSlope, reclassFieldSlope, remapWildcatSlope, "NODATA")

    # Save the output as raster with format TIF.
    outReclassifySlopeWildcat.save("outReclassifySlopeWildcat.tif")
    
SlopeClassifyWildcat() # Run the reclassification for Wildcat Helicopter requirements.

print('Wildcat slope reclassification complete.')

Reclassifying slope for Wildcat helicopter parameters.
Wildcat slope reclassification complete.


In [50]:
print('Reclassifying slope for Merlin helicopter parameters.')

def SlopeClassifyMerlin(): # This function reclassifies the 10m slope dataset from Degrees to 
                            # numerical values representing classes defined for the Merlin 
                            # Slope Requirements in the instruction manual document. 
    
    #Set processing environment, matching the parent folder location for the project, defined as home_folder in script header:
    env.workspace = home_folder
    
    # Set the extent of the processing environment using a feature class, defined by the user, set in the script header as process_extent: 
    arcpy.env.extent = process_extent
    
    # Set local variables:
    # Set location of the input 10m Slope dataset:
    inRasterMerlinSlope = "HLS_AOI_10m_Slope.tif"

    # Set the field or band of the raster which holds the numerical slope values in degrees:
    reclassFieldSlope = "Value"

    # Set the reclassification table as a list of tuples, based on the reclassification table in the instruction document.
    # 0 to 2 degrees = 3; 2 to 6 degrees = 2; 6 to 9 degrees = 1; 9 to 90 degrees = 0
    remapMerlinSlope = RemapRange([[0,2,3],[2,6,2],[6,9,1],[9,90,0]])
    
    # Execute Reclassify tool
    outReclassifySlopeMerlin = Reclassify(inRasterMerlinSlope, reclassFieldSlope, remapMerlinSlope, "NODATA")

    # Save the output as raster with format TIF.
    outReclassifySlopeMerlin.save("outReclassifySlopeMerlin.tif")
    
SlopeClassifyMerlin() # Run the reclassification for Wildcat Helicopter requirements.

print('Merlin slope reclassification complete.')

Reclassifying slope for Merlin helicopter parameters.
Merlin slope reclassification complete.


In [51]:
print('Reclassifying slope for Chinook helicopter parameters.')

def SlopeClassifyChinook(): # This function reclassifies the 10m slope dataset from Degrees to 
                            # numerical values representing classes defined for the Chinook 
                            # Slope Requirements in the instruction manual document. 
    
    #Set processing environment, matching the parent folder location for the project, defined as home_folder in script header:
    env.workspace = home_folder
    
    # Set the extent of the processing environment using a feature class, defined by the user, set in the script header as process_extent: 
    arcpy.env.extent = process_extent
    
    # Set local variables:
    # Set location of the input 10m Slope dataset:
    inRasterChinookSlope = "HLS_AOI_10m_Slope.tif"

    # Set the field or band of the raster which holds the numerical slope values in degrees:
    reclassFieldSlope = "Value"

    # Set the reclassification table as a list of tuples, based on the reclassification table in the instruction document.
    # 0 to 6 degrees = 3; 6 to 7 degrees = 2; 7 to 10 degrees = 1; 10 to 90 degrees = 0
    remapChinookSlope = RemapRange([[0,6,3],[6,7,2],[7,10,1],[10,90,0]])
    
    # Execute Reclassify tool
    outReclassifySlopeChinook = Reclassify(inRasterChinookSlope, reclassFieldSlope, remapChinookSlope, "NODATA")

    # Save the output as raster with format TIF.
    outReclassifySlopeChinook.save("outReclassifySlopeChinook.tif")
    
SlopeClassifyChinook() # Run the reclassification for Wildcat Helicopter requirements.

print('Chinook slope reclassification complete.')

Reclassifying slope for Chinook helicopter parameters.
Chinook slope reclassification complete.


The next three functions use the Multicriteria Analysis Tool to combine the values of Reclassified LULC with the three different Reclassified Slope per Helicopter rasters. This method allows the user to put an influence weighting on either LULC or Slope, if desired. For the default in this script, we will assume LULC and Slope have equal influence on the output.

In [52]:
print('Running Weighted Overlay for Wildcat helicopter parameters.')

def Wildcat_Multicriteria(): # This function sets up the multicriteria analysis process for the Wildcat Helicopter.
    
    #Set processing environment, matching the parent folder location for the project, defined as home_folder in script header:
    env.workspace = home_folder
    
    # Set the extent of the processing environment using a feature class, defined by the user, set in the script header as process_extent: 
    arcpy.env.extent = process_extent

    # Set local variables
    inRasterLULC = r"\outReclassifyLULC.tif"
    inRasterWildcatSlope = "outReclassifySlopeWildcat.tif"

    remapLULC = RemapValue([[0,"Restricted"],[1,1],[2,2],[3,3],["NODATA","NODATA"]]) # Define the remap table for the LULC classified raster. Value, remapped value pairs.
    remapWildcatSlope = RemapValue([[0,"Restricted"],[2,2],[3,3],["NODATA","NODATA"]]) # Define the remap table for the Wildcat Slope classified raster. Value, remapped value pairs.
    
    # Define the Weighted Overlay table for the Wildcat Helicopter, using the remap tables defined above:
    myWOTableWildcat = WOTable([[inRasterLULC, 50, "VALUE", remapLULC], # Assign 50% weighting to the LULC data.
                     [inRasterWildcatSlope, 50, "VALUE", remapWildcatSlope], # Assign 50% weighting to the Wildcat Slope data.
					          ], [1, 3, 1])    # Define the numerical scale for the output values. Minimum value, maximum value, and interval.

    # Execute WeightedOverlay
    outWeightedOverlayWildcat = WeightedOverlay(myWOTableWildcat)

    # Save the output
    outWeightedOverlayWildcat.save("outWeightedOverlayWildcat.tif")
    
Wildcat_Multicriteria() # Run the weighted multicriteria analysis for Wildcat Helicopter requirements.

print('Wildcat Weighted Overlay complete.')

Running Weighted Overlay for Wildcat helicopter parameters.
Wildcat Weighted Overlay complete.


In [53]:
print('Running Weighted Overlay for Merlin helicopter parameters.')

def Merlin_Multicriteria(): # This function sets up the multicriteria analysis process for the Merlin Helicopter.
    
    #Set processing environment, matching the parent folder location for the project, defined as home_folder in script header:
    env.workspace = home_folder
    
    # Set the extent of the processing environment using a feature class, defined by the user, set in the script header as process_extent: 
    arcpy.env.extent = process_extent

    # Set local variables
    inRasterLULC = r"\outReclassifyLULC.tif"
    inRasterMerlinSlope = "outReclassifySlopeMerlin.tif"

    remapLULC = RemapValue([[0,"Restricted"],[1,1],[2,2],[3,3],["NODATA","NODATA"]]) # Define the remap table for the LULC classified raster. Value, remapped value pairs.
    remapMerlinSlope = RemapValue([[0,"Restricted"],[1,1],[2,2],[3,3],["NODATA","NODATA"]]) # Define the remap table for the Merlin Slope classified raster. Value, remapped value pairs.
    
    # Define the Weighted Overlay table for the Merlin Helicopter, using the remap tables defined above:
    myWOTableMerlin = WOTable([[inRasterLULC, 50, "VALUE", remapLULC], # Assign 50% weighting to the LULC data.
                     [inRasterMerlinSlope, 50, "VALUE", remapMerlinSlope], # Assign 50% weighting to the Merlin Slope data.
					          ], [1, 3, 1])    # Define the numerical scale for the output values. Minimum value, maximum value, and interval.

    # Execute WeightedOverlay
    outWeightedOverlayMerlin = WeightedOverlay(myWOTableMerlin)

    # Save the output
    outWeightedOverlayMerlin.save("outWeightedOverlayMerlin.tif")
    
Merlin_Multicriteria() # Run the weighted multicriteria analysis for Merlin Helicopter requirements.

print('Merlin Weighted Overlay complete.')

Running Weighted Overlay for Merlin helicopter parameters.
Merlin Weighted Overlay complete.


In [54]:
print('Running Weighted Overlay for Wildcat helicopter parameters.')

def Chinook_Multicriteria(): # This function sets up the multicriteria analysis process for the Chinook Helicopter.
    
    #Set processing environment, matching the parent folder location for the project, defined as home_folder in script header:
    env.workspace = home_folder
    
    # Set the extent of the processing environment using a feature class, defined by the user, set in the script header as process_extent: 
    arcpy.env.extent = process_extent

    # Set local variables
    inRasterLULC = r"\outReclassifyLULC.tif"
    inRasterChinookSlope = "outReclassifySlopeChinook.tif"

    remapLULC = RemapValue([[0,"Restricted"],[1,1],[2,2],[3,3],["NODATA","NODATA"]]) # Define the remap table for the LULC classified raster. Value, remapped value pairs.
    remapChinookSlope = RemapValue([[0,"Restricted"],[1,1],[2,2],[3,3],["NODATA","NODATA"]]) # Define the remap table for the Chinook Slope classified raster. Value, remapped value pairs.
    
    # Define the Weighted Overlay table for the Chinook Helicopter, using the remap tables defined above:
    myWOTableChinook = WOTable([[inRasterLULC, 50, "VALUE", remapLULC], # Assign 50% weighting to the LULC data.
                     [inRasterChinookSlope, 50, "VALUE", remapChinookSlope], # Assign 50% weighting to the Chinook Slope data.
					          ], [1, 3, 1])    # Define the numerical scale for the output values. Minimum value, maximum value, and interval.

    # Execute WeightedOverlay
    outWeightedOverlayChinook = WeightedOverlay(myWOTableChinook)

    # Save the output
    outWeightedOverlayChinook.save("outWeightedOverlayChinook.tif")
    
Chinook_Multicriteria() # Run the weighted multicriteria analysis for Chinook Helicopter requirements.

print('Chinook Weighted Overlay complete.')

Running Weighted Overlay for Wildcat helicopter parameters.
Chinook Weighted Overlay complete.


In [55]:
print('Process finished successfully.')

Process finished successfully.
